# Быстрый старт: проверка локального стенда

Небольшой ноутбук, чтобы убедиться, что БД поднята, данные загружены,
и посчитать риск-таблицу прямо тут, без веб-приложения. Запускать после
`docker compose up -d` и `python db/import_data.py`.

In [ ]:
import os
os.environ.setdefault("DATABASE_URL", "postgresql://localhost:5432/soil_ml")
from sqlalchemy import create_engine
import pandas as pd
import geopandas as gpd

engine = create_engine(os.environ["DATABASE_URL"])
pd.read_sql("SELECT count(*) AS n_sites FROM sites", engine)

## Риск-таблица по складам (медиана Pb/As/Hg) — прямо из PostGIS

In [ ]:
query = '''
SELECT sam.site_id,
       count(DISTINCT sam.sample_id) AS n_points,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY m.value) FILTER (WHERE m.substance='Pb') AS pb_median,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY m.value) FILTER (WHERE m.substance='As') AS as_median,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY m.value) FILTER (WHERE m.substance='Hg') AS hg_median
FROM samples sam
JOIN measurements m ON m.sample_id = sam.sample_id
WHERE m.below_lod = FALSE
GROUP BY sam.site_id
ORDER BY pb_median DESC
'''
risk = pd.read_sql(query, engine)
risk

## Быстрая карта прямо в ноутбуке (без Flask)

In [ ]:
sites = gpd.read_postgis("SELECT * FROM sites", engine, geom_col="contour_geom")
sites.explore(column="site_id", legend=False, tiles="OpenStreetMap")

## Дальше

- Полная интерактивная карта с риск-цветами — через `webapp/app.py` (Flask + Leaflet).
- Пересчёт модели — `model_pipeline.py` из предыдущего пакета (можно переключить
  источник данных с CSV на `pd.read_sql` из этой же БД).
- Новые данные — добавляются через `db/import_data.py` или напрямую SQL-инсертами
  в `sites`/`samples`/`measurements`.